# Transformers: character-level language model
## Yoav Ram

We will see here the [**Transformer** architecture](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)).
Transformers are the basis of large language models like [GPT](https://en.wikipedia.org/wiki/Generative_pre-trained_transformer)--the "T" stands for "Transformer".

Here, we apply transformers to the same problem we applied RNN and GRU: text generation by pretraining a character level model.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" # jax compiler messages
os.environ["GRPC_VERBOSITY"] = "ERROR"  # if gRPC noise too
# os.environ['JAX_PLATFORMS'] = 'cpu' # force cpu

import jax
import jax.numpy as jnp
print('jax', jax.__version__, jax.default_backend())
import optax
import numpy as np          # real NumPy, for the data pipeline; jnp is JAX

import glob
import hashlib
import json
import pickle

# Data: the shared experiment

Like `GRU.ipynb`, this notebook does **not** define its own data. `RNN.ipynb` defines the experiment all three character-level notebooks share and writes it out; we load it.

This matters most here. A transformer has no hidden state to carry between windows, so the shared `state_policy` is `'none'` — every window starts cold, for every model. The earlier version of this comparison streamed the whole corpus through the RNN and GRU while giving the transformer short windows from a zero context, and then reported the three losses side by side. That is a difference in protocol, not in architecture.

If the artifacts are missing, the cell below **fails loudly** rather than re-deriving a split of its own.

In [ ]:
SPLIT_PATH = 'checkpoints/charlm_split.npz'
CONFIG_PATH = 'checkpoints/charlm_config.json'

for path in (SPLIT_PATH, CONFIG_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{path} is missing. Run RNN.ipynb first: it defines the shared '
            'experiment (split, budget, state policy, seed) that this notebook '
            'must reproduce exactly for the comparison to mean anything.')

with open(CONFIG_PATH) as f:
    CONFIG = json.load(f)

split = np.load(SPLIT_PATH, allow_pickle=False)
train_tokens, val_tokens = split['train'], split['val']
chars = [str(c) for c in split['vocab']]
vocab_size = len(chars)
int_to_char = dict(enumerate(chars))
char_to_int = {c: i for i, c in int_to_char.items()}

def array_hash(a):
    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()[:16]

if array_hash(val_tokens) != CONFIG['val_hash']:
    raise ValueError(
        'charlm_split.npz and charlm_config.json disagree: the validation array '
        f"hashes to {array_hash(val_tokens)} but the config records "
        f"{CONFIG['val_hash']}. Re-run RNN.ipynb to regenerate both together.")

print(f"corpus {CONFIG['corpus']}  |  vocabulary {vocab_size}")
print(f"train {len(train_tokens):,} characters  |  val {len(val_tokens):,}  "
      f"|  val_hash {CONFIG['val_hash']}")
print(f"context {CONFIG['context_length']}  |  batch {CONFIG['batch_size']}  "
      f"|  steps {CONFIG['steps']:,}  |  state_policy {CONFIG['state_policy']}  "
      f"|  seed {CONFIG['seed']}")

The split arrives already encoded as integers, one `uint8` per character. The helpers below are this notebook's own copies of the sampler, the bits-per-character metric and the evaluator — identical in behaviour to the ones in `RNN.ipynb` and `GRU.ipynb`, and checked against the same uniform-model assertion.

In [ ]:
def decode(ids):
    return ''.join(int_to_char[int(i)] for i in ids)

# The model below still consumes one-hot vectors. We materialise them per window,
# never for the whole corpus; the next revision of this notebook removes them.
def onehot(ids):
    return jax.nn.one_hot(jnp.asarray(ids, dtype=jnp.int32), vocab_size)

def onehot_decode(arr):
    return decode(np.asarray(arr).argmax(axis=-1))

def sample_batch(rng, tokens, batch_size, context_length):
    """Draw a batch of random windows. Returns (inputs, targets), both (B, T)."""
    hi = len(tokens) - context_length - 1
    start = rng.integers(0, hi, size=batch_size)
    idx = start[:, None] + np.arange(context_length + 1)[None, :]
    window = tokens[idx]
    return window[:, :-1].astype(np.int32), window[:, 1:].astype(np.int32)

def bits_per_char(total_nats, n_chars):
    return total_nats / n_chars / np.log(2)

def windows(tokens, context_length):
    n = (len(tokens) - 1) // context_length
    idx = np.arange(n * context_length).reshape(n, context_length)
    return tokens[idx].astype(np.int32), tokens[idx + 1].astype(np.int32)

def evaluate_bpc(logprob_fn, tokens, context_length, batch_size=64):
    """Mean bits per character, scored on the same windows as every other model,
    with nothing carried between windows (state_policy='none')."""
    x, y = windows(tokens, context_length)
    total_nats, n = 0.0, 0
    for i in range(0, len(x), batch_size):
        xb, yb = x[i:i + batch_size], y[i:i + batch_size]
        logp = np.asarray(logprob_fn(xb))
        picked = np.take_along_axis(logp, yb[:, :, None], axis=-1)[:, :, 0]
        total_nats += -picked.sum()
        n += picked.size
    return bits_per_char(total_nats, n)

def write_result(model, bpc, n_params=None, **extra):
    record = {'model': model, 'bpc': float(bpc), 'n_params': n_params,
              'manifest': dict(CONFIG), **extra}
    with open(f'checkpoints/charlm_result_{model}.json', 'w') as f:
        json.dump(record, f, indent=2)
    return record

# the evaluator must reproduce the known answer for a uniform model
uniform = lambda x: np.full(x.shape + (vocab_size,), -np.log(vocab_size))
assert np.isclose(evaluate_bpc(uniform, val_tokens, CONFIG['context_length']),
                  np.log2(vocab_size))
print('evaluator agrees with log2(V) on a uniform model')

# Text transformer
Unlike RNN and GRU which process sequences step by step maintaining a hidden state, the Transformer processes the entire sequence at once using **self-attention** — each position can directly attend to all previous positions.

## Self-attention

For each position $i$ in the input sequence $x$, we compute **query**, **key**, and **value** vectors:
$$
q_i = W^q x_i, \quad k_j = W^k x_j, \quad v_j = W^v x_j
$$

The **attention weight** of position $i$ to position $j$ is:
$$
w_{ij} = \text{softmax}_j\left(\frac{q_i \cdot k_j}{\sqrt{d_k}}\right)
$$

The output for position $i$ is a weighted sum of values: $z_i = \sum_j w_{ij} v_j$.

A **causal mask** ensures position $i$ only attends to $j \le i$, preserving the autoregressive property (just as in RNN/GRU, character $i$ depends on $j < i$ but not $j > i$).

In **multi-head** attention, we split $q$, $k$, $v$ into multiple heads that attend independently, then concatenate their outputs. This allows the model to attend to different types of relationships simultaneously.


In [ ]:
def self_attention(x, W_q, W_k, W_v, W_o, n_heads):
    seq_len, d_model = x.shape
    d_head = d_model // n_heads
    
    Q = x @ W_q  # (seq_len, d_model)
    K = x @ W_k
    V = x @ W_v
    
    # reshape to (n_heads, seq_len, d_head) - no batch dimension
    Q = Q.reshape(seq_len, n_heads, d_head).transpose(1, 0, 2)
    K = K.reshape(seq_len, n_heads, d_head).transpose(1, 0, 2)
    V = V.reshape(seq_len, n_heads, d_head).transpose(1, 0, 2)
    
    # attention weights
    w_logits = (Q @ K.transpose(0, 2, 1)) / jnp.sqrt(d_head)  # (n_heads, seq_len, seq_len)
    # causal mask
    mask = jnp.tril(jnp.ones((seq_len, seq_len)))
    w_logits = w_logits - 1e10 * (1 - mask)
    w = jax.nn.softmax(w_logits, axis=-1)
    
    # weighted sum of values
    z = w @ V  # (n_heads, seq_len, d_head)
    # concatenate heads
    z = z.transpose(1, 0, 2).reshape(seq_len, d_model)
    
    return z @ W_o

## Transfomer blocks

Each transformer block applies:
1. Layer normalization → multi-head self-attention → residual connection
2. Layer normalization → feed-forward network → residual connection

In [ ]:
def layer_norm(x, eps=1e-6):
    mean = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.mean((x - mean) ** 2, axis=-1, keepdims=True)
    return (x - mean) / jnp.sqrt(var + eps)

In [ ]:
def transformer_block(x, params, n_heads):
    # self-attention with pre-norm and residual
    x_norm = layer_norm(x)
    x = x + self_attention(x_norm, params['W_q'], params['W_k'],  params['W_v'], params['W_o'], n_heads)
    # feed-forward network with pre-norm and residual connection
    x_norm = layer_norm(x)
    h = jax.nn.relu(x_norm @ params['W1'] + params['b1'])
    x = x + h @ params['W2'] + params['b2']
    return x

def init_layer_params(key, d_model, d_ff):
    keys = jax.random.split(key, 6)
    return {
        'W_q': jax.random.normal(keys[0], (d_model, d_model)) * 0.02,
        'W_k': jax.random.normal(keys[1], (d_model, d_model)) * 0.02,
        'W_v': jax.random.normal(keys[2], (d_model, d_model)) * 0.02,
        'W_o': jax.random.normal(keys[3], (d_model, d_model)) * 0.02,
        'W1': jax.random.normal(keys[4], (d_model, d_ff)) * 0.02,
        'b1': jnp.zeros((d_ff,)),
        'W2': jax.random.normal(keys[5], (d_ff, d_model)) * 0.02,
        'b2': jnp.zeros((d_model,)),
    }


## Transformer classifier

The full model stacks multiple transformer blocks. Unlike RNN and GRU where the input weight matrix $W_x^h$ implicitly serves as an embedding (since $W_x^h \cdot \text{onehot}(c)$ simply selects column $c$ of the matrix), the transformer has no per-step input weight matrix — it applies the same attention mechanism to all positions.
It therefore needs an explicit **learned token embedding** to convert characters to continuous vectors.
Similarly, it uses a **learned positional embedding** to encode position information, since the attention mechanism is permutation-invariant and has no inherent notion of order.

The final layer is a classifier head that predicts the next character.

In [ ]:
def transformer_model(params, x, n_heads):
    # convert from one-hot to integer indices
    indices = x.argmax(axis=1)
    seq_len = indices.shape[0]
    # token embedding + positional embedding
    x = params['token_emb'][indices] + params['pos_emb'][:seq_len]
    # transformer layers
    for layer_params in params['layers']:
        x = transformer_block(x, layer_params, n_heads)
    # final layer norm and classifier head that outputs logits
    x = layer_norm(x)
    logits = x @ params['W_out'] + params['b_out']
    return logits

def init_params(key, vocab_size, seq_length, d_model, d_ff, n_layers):
    keys = jax.random.split(key, n_layers + 3)
    return {
        'token_emb': jax.random.normal(keys[0], (vocab_size, d_model)) * 0.02,
        'pos_emb': jax.random.normal(keys[1], (seq_length, d_model)) * 0.02,
        'layers': [init_layer_params(keys[2 + i], d_model, d_ff) for i in range(n_layers)],
        'W_out': jax.random.normal(keys[-1], (d_model, vocab_size)) * 0.02,
        'b_out': jnp.zeros((vocab_size,)),
    }

# Loss function

The loss function is categorical cross-entropy (negative log-likelihood).
We use `log_softmax` for numerical stability.

In [ ]:
seq_length = CONFIG['context_length']
d_model = 128
d_ff = 512
n_heads = 4
n_layers = 5

key = jax.random.key(0)
params = init_params(key, vocab_size, seq_length, d_model, d_ff, n_layers)

# one-hot only for the window this test needs, not the whole corpus
x, y = onehot(train_tokens[:seq_length]), onehot(train_tokens[1:seq_length + 1])
logits = transformer_model(params, x, n_heads)
assert logits.shape == (seq_length, vocab_size)
print("logits shape:", logits.shape)

The NLL (negative log-likelihood, cross-entropy) for a single step is
$$
NLL = -\sum_{k=0}^{V}{x_{t+1,k} \log{\hat{y}_{t,k}}}
$$
where $k$ runs over all $V$ characters.
The total NLL is computed by averaging over all positions $t$.

In [ ]:
def NLL(params, x, y):
    logits = transformer_model(params, x, n_heads)
    log_probs = jax.nn.log_softmax(logits)
    loss = -(y * log_probs).sum() / x.shape[0]
    return loss

loss = NLL(params, x, y)
print(loss)

## Automatic differentiation with JAX

We use JAX's automatic differentiation to compute gradients. [`jax.value_and_grad`](https://jax.readthedocs.io/en/latest/jax-101/01-jax-basics.html#value-and-grad) returns both the loss value and the gradient of the loss with respect to the parameters.

In [ ]:
backprop = jax.value_and_grad(NLL)

loss, grads = backprop(params, x, y)
for k in params:
    if k == 'layers':
        for layer_params, layer_grads in zip(params[k], grads[k]):
            for p_name in layer_params:
                assert layer_params[p_name].shape == layer_grads[p_name].shape
    else:
        assert params[k].shape == grads[k].shape

# Adam optimizer with Optax

We can use a JAX implementation of the Adam optimizer from the [Optax](https://optax.readthedocs.io/) library.
We first create the optimizer and initialize its state.

In [ ]:
optimizer = optax.adam(learning_rate=0.001) # 0.001 is the default from Kingma et al 2014
opt_state = optimizer.init(params)

We then use the optimizer to compute the updates, and apply them.

In [ ]:
loss, grads = backprop(params, x, y)
updates, opt_state = optimizer.update(grads, opt_state, params)
params = optax.apply_updates(params, updates)

# JITing the training step

We write a function that does all this, and pass it to `jax.jit`, which [just-in-time compiles the function](https://jax.readthedocs.io/en/latest/jax-101/02-jitting.html) so it can be executed efficiently in XLA.

In [ ]:
@jax.jit
def update_params(params, opt_state, x, y):
    loss, grads = backprop(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

In [ ]:
%timeit update_params(params, opt_state, x, y)

In [ ]:
params, opt_state, loss = update_params(params, opt_state, x, y)
print(loss)
params, opt_state, loss = update_params(params, opt_state, x, y)
print(loss)

# Sampling from the network

Instead of a `predict` function, we have a `sample` function, which, given the parameters and the number of samples we want, produces a sample of text from the network.

It does so by drawing a random seed for $x_0$ and drawing $x_t$ for $t>0$ from the distribution given by $\widehat y_t$.

Unlike RNN/GRU which maintain a hidden state, the transformer must re-process the entire sequence at each step to generate the next character., which makes sampling - or as it is commonly called, **inference** - scale quadratically.

In practice, this is mitigated by caching the key-value (KV) projections from previous tokens, so only the new token's attention computations are added incrementally rather than recomputed from scratch. For very long contexts (sequences), the cost of sampling is still quadratic, and much research and effort goes into the development of more efficient sampling algorithms.

In [ ]:
def sample(params, num_samples, key):
    x = jnp.zeros((num_samples, vocab_size), dtype=float)
    key, subkey = jax.random.split(key)
    seed_char = jax.random.choice(subkey, vocab_size)
    x = x.at[0, seed_char].set(1)
    
    for t in range(1, num_samples):
        # sliding window: only use the last seq_length characters as context
        start = max(0, t - seq_length)
        logits = transformer_model(params, x[start:t], n_heads)[-1]
        yhat = jax.nn.softmax(logits)
        # draw from output distribution
        key, subkey = jax.random.split(key)
        i = jax.random.choice(subkey, vocab_size, p=yhat)
        x = x.at[t, i].set(1)
    return onehot_decode(x)

print(sample(params, 100, jax.random.key(1)))

# Training the network

We setup the training - the sequence length, the number of batches, parameter initialization, Adam optimizer.

In [ ]:
seq_length = CONFIG['context_length']
max_batches = CONFIG['steps']
pos = 0
batch = 0
losses = []
key = jax.random.key(CONFIG['seed'])

d_model = 128
d_ff = 512
n_heads = 4
n_layers = 5

params = init_params(key, vocab_size, seq_length, d_model, d_ff, n_layers)

backprop = jax.value_and_grad(NLL)

# Warmup + cosine decay over the full budget, with gradient clipping.
# Same schedule as RNN.ipynb and GRU.ipynb: the learning rate is part of the
# shared experiment, not a per-model tuning knob.
schedule = optax.warmup_cosine_decay_schedule(
    init_value=CONFIG['init_lr'],
    peak_value=CONFIG['peak_lr'],
    warmup_steps=CONFIG['warmup_steps'],
    decay_steps=CONFIG['steps'],
)
optimizer = optax.chain(
    optax.clip_by_global_norm(CONFIG['grad_clip']),
    optax.adam(learning_rate=schedule),
)
opt_state = optimizer.init(params)

Now we can train the Transformer!

In [ ]:
%%time
# NOTE: still one window per step, walking a cursor through the corpus. The next
# revision replaces this with the batched sampler defined above.
n_train = len(train_tokens)
report_every = max(1, max_batches // 10)

while batch <= max_batches:
    if pos + seq_length + 1 >= n_train:
        pos = 0

    x = onehot(train_tokens[pos : pos + seq_length])
    y = onehot(train_tokens[pos + 1 : pos + seq_length + 1])
    pos += seq_length

    params, opt_state, loss = update_params(params, opt_state, x, y)
    losses.append(loss)

    if batch % report_every == 0:
        print('batch {:d}, loss {:.6f}, pos {}'.format(batch, loss, pos))
        print()

        with open("checkpoints/transformer-jax-params-{}.pkl".format(batch), 'wb') as file:
            pickle.dump(params, file)

        key, subkey = jax.random.split(key)
        sample_text = sample(params, 200, subkey)
        print(sample_text)
        print('-'*80)

    batch += 1

In [ ]:
with open("checkpoints/transformer-jax-params-{}.pkl".format(batch), 'wb') as file:
    pickle.dump(params, file)

# Load parameters from specific batch

In [ ]:
paths = sorted(glob.glob("checkpoints/transformer-jax-params-*.pkl"))
if not paths:
    raise FileNotFoundError("No Transformer checkpoint files found in checkpoints/")

latest = paths[-1]
print("Loading checkpoint:", latest)
with open(latest, 'rb') as file:
    params = pickle.load(file)

In [ ]:
print(sample(params, 500, jax.random.key(21)))

# Results so far

The last notebook in the sequence, so this table is the complete comparison — provided you have run `RNN.ipynb` and `GRU.ipynb`. Any model you have not run is named as `not run` rather than omitted, and the table refuses to render if two results disagree about the experiment they ran under.

Read it with the state policy in mind: all three models were trained and scored on cold 128-character windows, which is the only protocol a transformer can implement, and which costs the recurrent models the long-range context they would normally carry.

In [ ]:
EXPECTED_MODELS = ['unigram', 'bigram', 'rnn', 'gru', 'transformer']

def results_table():
    """Print every charlm result written so far, checking they are comparable."""
    found = {}
    for path in sorted(glob.glob('checkpoints/charlm_result_*.json')):
        with open(path) as f:
            r = json.load(f)
        found[r['model']] = r

    if not found:
        print('No results yet.')
        return

    reference = next(iter(found.values()))['manifest']
    keys = ['val_hash', 'state_policy', 'context_length', 'steps', 'batch_size', 'seed']
    for name, r in found.items():
        for k in keys:
            if r['manifest'].get(k) != reference.get(k):
                raise ValueError(
                    f"'{name}' ran under a different experiment: {k} is "
                    f"{r['manifest'].get(k)!r}, expected {reference.get(k)!r}. "
                    'Re-run it against the current charlm_config.json.')

    print(f"corpus {reference['corpus']} | context {reference['context_length']} | "
          f"state_policy {reference['state_policy']} | seed {reference['seed']} (single run)")
    print()
    print(f"{'model':<14}{'bpc':>8}{'params':>12}")
    print('-' * 34)
    for name in EXPECTED_MODELS:
        if name in found:
            r = found[name]
            p = r.get('n_params')
            print(f"{name:<14}{r['bpc']:>8.3f}{('' if p is None else f'{p:,}'):>12}")
        else:
            print(f"{name:<14}{'not run':>8}{'':>12}")

results_table()

# References

- [Vaswani et al. 2017](http://arxiv.org/abs/1706.03762): _Attention Is All You Need_, the fundamental paper on transformers.


# Colophon
This notebook was written by [Yoav Ram](http://python.yoavram.com).

This work is licensed under a [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) International License.

![Python logo](https://www.python.org/static/community_logos/python-logo.png)